**Autor:** Alan Sastre  
**GitHub:** [github.com/alansastre/genai-fundamentos](https://github.com/alansastre/genai-fundamentos)

---

## El flujo completo en 5 pasos

Antes de entrar en detalle, este es el proceso que sigue un LLM desde que recibe texto hasta que genera una respuesta:

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart LR
    T["Texto"] --> TOK["Tokenizar"]
    TOK --> EMB["Embeddings"]
    EMB --> BLK["Bloques<br/>Transformer"]
    BLK --> PRED["Predecir"]
    PRED --> OUT["Texto generado"]
    OUT -.->|"Repetir"| TOK

    style BLK fill:#9f7aea,color:#fff
    style OUT fill:#4ade80,color:#fff
```

| Paso | Qué hace | Por qué es necesario |
|------|----------|----------------------|
| **Tokenizar** | Divide el texto en fragmentos | Los modelos trabajan con números, no con letras |
| **Embeddings** | Convierte fragmentos en vectores | Los vectores capturan el significado |
| **Bloques Transformer** | Atención + procesamiento en capas | Permite entender contexto y extraer significado |
| **Predecir** | Calcula probabilidad de cada palabra | Determina qué palabra viene después |
| **Repetir** | Añade la palabra y vuelve a empezar | Genera texto palabra a palabra |

> El proceso se repite: cada token generado se añade a la secuencia y el modelo predice el siguiente, hasta completar la respuesta.

## Paso 1: Tokenización

El modelo no trabaja con palabras ni caracteres. El primer paso es la **tokenización**: dividir el texto en fragmentos llamados *tokens*.

Un token puede ser:
- Una palabra completa: "casa"
- Una subpalabra: "ción", "pre"
- Un carácter en casos especiales

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart LR
    T["El gato duerme"] --> TOK[Tokenizador]
    TOK --> T1["El"]
    TOK --> T2["gato"]
    TOK --> T3["duer"]
    TOK --> T4["me"]

    style TOK fill:#9f7aea,color:#fff
```

El modelo tiene un **vocabulario fijo** de 30,000 a 100,000 tokens. Cada token tiene un número único (ID). La frase "El gato duerme" se convierte en algo como `[128, 4521, 892, 45]`.

> La tokenización permite manejar cualquier texto, incluyendo palabras nuevas, dividiéndolas en partes conocidas.

## Paso 2: Embeddings

Cada ID de token se convierte en un **vector**: una lista de números que representa su significado. Estos vectores tienen cientos o miles de dimensiones.

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart TB
    subgraph tokens["Tokens"]
        ID1["128"]
        ID2["4521"]
    end

    subgraph vectores["Vectores"]
        V1["[0.2, -0.5, 0.8, ...]"]
        V2["[0.1, 0.3, -0.2, ...]"]
    end

    ID1 --> V1
    ID2 --> V2

    style vectores fill:#e8f4f8
```

Lo importante de los embeddings es que **capturan significado**:
- Palabras similares tienen vectores cercanos
- "Rey" y "reina" están próximos
- "Perro" y "gato" más cerca entre sí que "perro" y "democracia"

Además, se añade **información de posición** para que el modelo sepa el orden de las palabras. Sin esto, "perro muerde hombre" y "hombre muerde perro" serían iguales.

## Paso 3: Bloques Transformer

El corazón del modelo son los **bloques Transformer**. Cada bloque tiene dos componentes que trabajan en secuencia:

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart LR
    subgraph bloque["Bloque Transformer"]
        ATT["Atención"] --> FF["Procesamiento"]
    end
    
    IN["Entrada"] --> bloque --> OUT["Salida"]
    
    style ATT fill:#9f7aea,color:#fff
    style FF fill:#38bdf8,color:#fff
```

| Componente | Qué hace | Pregunta que responde |
|------------|----------|----------------------|
| **Atención** | Cada palabra "mira" a las demás | "¿Qué palabras son relevantes para esta?" |
| **Procesamiento** | Transforma la información combinada | "¿Qué significa esa combinación?" |

### Atención: entender el contexto

El mecanismo de **atención** permite que cada palabra mire a todas las anteriores para entender el contexto.

Para la frase "El banco cerca del río estaba vacío":
- Cuando procesa "vacío", el modelo atiende a "banco" y "río"
- Entiende que "banco" se refiere a un asiento, no a una entidad financiera
- Asigna más "peso" a las palabras relevantes

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart TB
    subgraph atencion["Atención para 'vacío'"]
        W1["El: 0.05"]
        W2["banco: 0.35"]
        W3["cerca: 0.10"]
        W4["del: 0.05"]
        W5["río: 0.30"]
        W6["estaba: 0.15"]
    end

    style W2 fill:#9f7aea,color:#fff
    style W5 fill:#9f7aea,color:#fff
```

El modelo tiene **múltiples "cabezas" de atención** que trabajan en paralelo, cada una especializada en detectar diferentes tipos de relaciones (sintácticas, semánticas, referencias, etc.).

> La atención es lo que permite al modelo entender que la misma palabra tiene significados diferentes según el contexto.

### Procesamiento: transformar la información

Después de la atención, una **red feed-forward** (también llamada MLP) procesa la información. Mientras la atención decide qué es relevante, el procesamiento transforma y combina esa información para extraer significado.

### Capas apiladas

Un LLM apila **decenas o cientos de bloques** Transformer. Cada bloque refina la representación, añadiendo más contexto y abstracción:

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart TB
    EMB["Embeddings"] --> B1["Bloque 1"]
    B1 --> B2["Bloque 2"]
    B2 --> BN["..."]
    BN --> BF["Bloque N"]
    BF --> PRED["Predicción"]

    style B1 fill:#9f7aea,color:#fff
    style B2 fill:#9f7aea,color:#fff
    style BF fill:#9f7aea,color:#fff
```

> Las primeras capas capturan patrones simples (gramática, sintaxis). Las capas profundas capturan conceptos abstractos (significado, razonamiento).

## Paso 4: Predicción

Tras procesar todo el contexto, el modelo predice el **siguiente token**. Toma el vector del último token y calcula una probabilidad para cada palabra del vocabulario.

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart LR
    V["Vector final"] --> PROJ["Proyección"]
    PROJ --> PROB["Probabilidades<br/>50,000 valores"]
    PROB --> SEL["Selección"]
    SEL --> TOKEN["Palabra elegida"]

    style PROB fill:#9f7aea,color:#fff
    style TOKEN fill:#4ade80,color:#fff
```

Si el vocabulario tiene 50,000 tokens, el modelo produce 50,000 probabilidades:

| Token | Probabilidad |
|-------|--------------|
| "feliz" | 0.15 |
| "contento" | 0.12 |
| "triste" | 0.08 |
| ... | ... |

### El parámetro temperatura

El modelo no siempre elige la palabra más probable. El parámetro **temperatura** controla esto:

| Temperatura | Comportamiento |
|-------------|----------------|
| **Baja (0.0-0.3)** | Respuestas predecibles, coherentes |
| **Media (0.5-0.7)** | Balance entre coherencia y creatividad |
| **Alta (0.8-1.0)** | Respuestas más creativas pero menos predecibles |

> La temperatura es el parámetro más importante que puedes ajustar al usar un LLM. Valores bajos para código y datos, valores altos para texto creativo.

## Paso 5: Generación iterativa

Una vez seleccionado un token, se añade a la secuencia y el proceso se repite:

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart TB
    P1["El gato"] --> M1["Modelo"] --> T1["duerme"]
    P2["El gato duerme"] --> M2["Modelo"] --> T2["en"]
    P3["El gato duerme en"] --> M3["Modelo"] --> T3["el"]

    style M1 fill:#9f7aea,color:#fff
    style M2 fill:#9f7aea,color:#fff
    style M3 fill:#9f7aea,color:#fff
```

Este proceso continúa hasta que:
- El modelo genera un token especial de fin
- Se alcanza el límite máximo de tokens
- El usuario interrumpe la generación

> Cada token generado influye en los siguientes. Por eso pequeños cambios en el prompt pueden producir respuestas muy diferentes.

## Resumen visual

```mermaid
%%{init: {'theme': 'default'}}%%
flowchart TB
    subgraph entrada["1. Entrada"]
        TEXT["El gato duerme"]
    end

    subgraph proceso["2. Proceso interno"]
        TOK["Tokenizar: [128, 4521, 892, 45]"]
        EMB["Embeddings: vectores con significado"]
        BLK["Bloques Transformer: atención + procesamiento"]
        PRED["Predicción: probabilidad por palabra"]
    end

    subgraph salida["3. Salida"]
        OUT["en (siguiente palabra más probable)"]
    end

    entrada --> TOK --> EMB --> BLK --> PRED --> salida
    salida -.->|"Repetir hasta terminar"| TOK

    style BLK fill:#9f7aea,color:#fff
    style salida fill:#4ade80,color:#fff
```

Lo que debes recordar:

1. **Tokenización**: el texto se divide en fragmentos con IDs numéricos
2. **Embeddings**: cada ID se convierte en un vector que captura significado
3. **Bloques Transformer**: cada bloque tiene atención (qué es relevante) y procesamiento (qué significa)
4. **Predicción**: calcula probabilidades y elige la siguiente palabra
5. **Iteración**: repite el proceso hasta completar la respuesta

> No necesitas entender los detalles matemáticos para usar LLMs efectivamente. Lo importante es saber que predicen texto palabra a palabra, basándose en el contexto previo.